# 05 - Decision Tree Regressor para PM2.5

Implementacion del modelo de arbol de decision para predecir PM2.5 en Lima.

## 1. Librerias y rutas

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_DIR / 'data' / 'processed' / 'datos_modelo.csv'
RAW_PATH = PROJECT_DIR / 'data' / 'raw' / 'datos_horarios_contaminacion_lima.xlsx'
MODELS_DIR = PROJECT_DIR / 'models'
REPORTS_DIR = PROJECT_DIR / 'reports'
IMAGES_DIR = REPORTS_DIR / 'imagenes'
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'PM 2.5'
FEATURES = ['PM 10', 'SO2', 'NO2', 'O3', 'CO', 'ANO', 'MES', 'DIA', 'HORA', 'ESTACION']
NUMERIC_FEATURES = ['PM 10', 'SO2', 'NO2', 'O3', 'CO', 'ANO', 'MES', 'DIA', 'HORA']
CATEGORICAL_FEATURES = ['ESTACION']

def one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

def clean_if_needed():
    if DATA_PATH.exists():
        return pd.read_csv(DATA_PATH)

    df = pd.read_excel(RAW_PATH)
    pollutants = ['PM 10', 'PM 2.5', 'SO2', 'NO2', 'O3', 'CO']
    for col in pollutants:
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.drop_duplicates(subset=['CODIGO ESTACION', 'ANO', 'MES', 'DIA', 'HORA']).copy()
    df = df.sort_values(['CODIGO ESTACION', 'ANO', 'MES', 'DIA', 'HORA'])

    for col in ['PM 10', 'SO2', 'NO2', 'O3', 'CO']:
        df[col] = df.groupby('CODIGO ESTACION')[col].transform(lambda s: s.interpolate(limit_direction='both'))
        df[col] = df[col].fillna(df.groupby(['CODIGO ESTACION', 'HORA'])[col].transform('median'))
        df[col] = df[col].fillna(df[col].median())

    df = df.dropna(subset=[TARGET] + FEATURES).copy()
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df[FEATURES + [TARGET]].to_csv(DATA_PATH, index=False)
    return df[FEATURES + [TARGET]]

from sklearn.tree import DecisionTreeRegressor

## 2. Carga de datos limpios y separacion train/test

In [ ]:
df = clean_if_needed()
print(df.shape)
df.head()

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)
print('Entrenamiento:', X_train.shape)
print('Prueba:', X_test.shape)


## 3. Pipeline de preprocesamiento y modelo

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
        ]), NUMERIC_FEATURES),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', one_hot_encoder()),
        ]), CATEGORICAL_FEATURES),
    ],
    verbose_feature_names_out=False,
)

tree_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(max_depth=8, min_samples_leaf=100, random_state=42)),
])
tree_model

## 4. Entrenamiento

In [ ]:
tree_model.fit(X_train, y_train)
y_pred = tree_model.predict(X_test)
print('Modelo Decision Tree entrenado')

## 5. Evaluacion

In [ ]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2': r2_score(y_true, y_pred),
    }

metrics_tree = regression_metrics(y_test, y_pred)
pd.DataFrame([metrics_tree])

## 6. Importancia de variables

In [ ]:
feature_names = tree_model.named_steps['preprocessor'].get_feature_names_out()
importances = tree_model.named_steps['model'].feature_importances_
importance_df = pd.DataFrame({'Variable': feature_names, 'Importancia': importances})
importance_df = importance_df.sort_values('Importancia', ascending=False).head(12)
importance_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(importance_df['Variable'].iloc[::-1], importance_df['Importancia'].iloc[::-1])
plt.title('Importancia de variables - Decision Tree')
plt.show()

## 7. Guardado del modelo

In [ ]:
joblib.dump(tree_model, MODELS_DIR / 'decision_tree_regressor.pkl')
print(MODELS_DIR / 'decision_tree_regressor.pkl')